# JMIR multilingual evidence-selection experiment — main evaluation

Runs the frozen 125-case experiment: 3 languages × 3 evidence conditions × 2 open-weight VLMs = **2250 trials**. Every response is appended immediately to Google Drive. Re-running safely skips completed trial IDs.

Use a **T4 GPU runtime**. Run models sequentially; a second Colab session is neither needed nor recommended.


## 0. Mount Drive, verify the frozen casebook, and create the fixed trial schedule


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, json, os, random, re, subprocess, time, urllib.request
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK_DIR = ROOT / 'casebooks'
RESULTS = ROOT / 'results' / 'main_evaluation'
RESULTS.mkdir(parents=True, exist_ok=True)

CASEBOOK = CASEBOOK_DIR / 'evaluation_cases_frozen.csv'
HASHFILE = CASEBOOK_DIR / 'evaluation_cases_frozen.csv.sha256'
assert CASEBOOK.exists(), f'Missing: {CASEBOOK}'
assert HASHFILE.exists(), f'Missing: {HASHFILE}'

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

expected_hash = HASHFILE.read_text(encoding='utf-8').strip().split()[0]
actual_hash = sha256_file(CASEBOOK)
assert actual_hash == expected_hash, f'Frozen casebook hash mismatch: {actual_hash} != {expected_hash}'

cases = pd.read_csv(CASEBOOK, dtype=str, keep_default_na=False).sort_values('case_id').reset_index(drop=True)
assert len(cases) == 125
assert cases['dataset'].value_counts().to_dict() == {'vqa_rad':100, 'slake':25}
assert cases['reference_answer'].value_counts().to_dict() == {'no':63, 'yes':62}
assert cases['case_id'].is_unique and cases['image_key'].is_unique and cases['image_path'].is_unique

SEED = 20260921
LANGUAGES = ['en','id','ms']
CONDITIONS = ['image_only','concordant','conflicting']

schedule = [
    {'case_id':row.case_id, 'language':language, 'condition':condition}
    for _, row in cases.iterrows()
    for language in LANGUAGES
    for condition in CONDITIONS
]
random.Random(SEED).shuffle(schedule)
schedule = pd.DataFrame(schedule)
assert len(schedule) == 1125 and not schedule.duplicated().any()
SCHEDULE = RESULTS / 'fixed_trial_schedule.csv'
schedule_bytes = schedule.to_csv(index=False).encode('utf-8')
schedule_sha = hashlib.sha256(schedule_bytes).hexdigest()
if SCHEDULE.exists() and SCHEDULE.read_bytes() != schedule_bytes:
    raise RuntimeError('Existing trial schedule differs from the fixed protocol schedule.')
SCHEDULE.write_bytes(schedule_bytes)
(RESULTS/'fixed_trial_schedule.csv.sha256').write_text(f'{schedule_sha}  {SCHEDULE.name}\n', encoding='utf-8')

print('Frozen casebook verified:', len(cases), 'cases')
print('Casebook SHA256:', actual_hash)
print('Fixed schedule:', len(schedule), 'trials per model')
print('Schedule SHA256:', schedule_sha)


Mounted at /content/drive
Frozen casebook verified: 125 cases
Casebook SHA256: c1617014f21be02ae1ec78ebb3cf6e862429a88552c0bc25c721b1fa07a9ab46
Fixed schedule: 1125 trials per model
Schedule SHA256: 60c491e1ca05633adc74a21de10d7bda52ce12c9d6db4c613ca5daabdc38c86c


## 1. Confirm the GPU

If this reports no NVIDIA GPU, change the Colab runtime to T4 before continuing.


In [2]:
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. Select a T4 GPU runtime and reconnect.')
print(gpu.stdout)


Mon Sep 21 15:14:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install and start Ollama

The asset selector deliberately excludes MLX and chooses the standard Linux AMD64 archive.


In [3]:
import platform, shutil

def find_ollama():
    found = shutil.which('ollama')
    if found: return found
    for p in ['/usr/local/bin/ollama','/usr/bin/ollama']:
        if Path(p).is_file(): return p
    return None

OLLAMA = find_ollama()
if OLLAMA is None:
    request = urllib.request.Request(
        'https://api.github.com/repos/ollama/ollama/releases/latest',
        headers={'User-Agent':'Google-Colab'}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        release = json.load(response)
    preferred = ['ollama-linux-amd64.tar.zst','ollama-linux-amd64.tgz']
    asset = next((a for name in preferred for a in release.get('assets',[]) if a['name'].lower()==name), None)
    if asset is None:
        names = [a['name'] for a in release.get('assets',[]) if 'linux' in a['name'].lower()]
        raise RuntimeError(f'Standard Linux AMD64 Ollama asset not found. Available: {names}')
    archive = Path('/content') / asset['name']
    print('Ollama release:', release['tag_name'])
    print('Downloading:', asset['name'])
    subprocess.run(['curl','-fL','--retry','5','--retry-delay','5','--connect-timeout','30','-o',str(archive),asset['browser_download_url']], check=True)
    if archive.name.endswith('.tar.zst'):
        subprocess.run(['apt-get','update','-qq'], check=True)
        subprocess.run(['apt-get','install','-y','-qq','zstd'], check=True)
        subprocess.run(['tar','--use-compress-program=unzstd','-xf',str(archive),'-C','/usr'], check=True)
    else:
        subprocess.run(['tar','-xzf',str(archive),'-C','/usr'], check=True)
    OLLAMA = find_ollama()
    if OLLAMA is None:
        hits = subprocess.run(['find','/usr','-type','f','-name','ollama'],capture_output=True,text=True).stdout.splitlines()
        if hits:
            target = Path('/usr/local/bin/ollama')
            if target.exists() or target.is_symlink(): target.unlink()
            target.symlink_to(Path(hits[0]))
            OLLAMA = str(target)
    if OLLAMA is None:
        raise RuntimeError('Ollama archive extracted but executable was not found.')

def ollama_alive():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

if not ollama_alive():
    ollama_log = open('/content/ollama_server.log','a')
    ollama_process = subprocess.Popen([OLLAMA,'serve'],stdout=ollama_log,stderr=subprocess.STDOUT)
    for _ in range(90):
        if ollama_alive(): break
        time.sleep(1)
if not ollama_alive():
    log = Path('/content/ollama_server.log').read_text(errors='replace')[-3000:]
    raise RuntimeError('Ollama server did not start. Log:\n'+log)
print(subprocess.run([OLLAMA,'--version'],capture_output=True,text=True).stdout.strip())
print('Ollama server ready:', OLLAMA)


Ollama release: v0.34.2
Downloading: ollama-linux-amd64.tar.zst
ollama version is 0.34.2
Ollama server ready: /usr/bin/ollama


## 3. Locked prompts, option counterbalancing, and image resolution

These definitions are identical to the successful pilot. Do not edit them after starting the main run.


In [27]:
QUESTION_COL = {'en':'question_en','id':'question_id','ms':'question_ms'}
PROP_COL = {'en':'proposition_en','id':'proposition_id','ms':'proposition_ms'}
YESNO = {'en':('Yes','No'),'id':('Ya','Tidak'),'ms':('Ya','Tidak')}
INTRO = {
    'en':'Examine the medical image and answer the question.',
    'id':'Periksa gambar medis dan jawab pertanyaannya.',
    'ms':'Periksa imej perubatan dan jawab soalan.'
}
CHOOSE = {
    'en':'Choose exactly one option. Reply with A or B only.',
    'id':'Pilih tepat satu opsi. Jawab hanya dengan A atau B.',
    'ms':'Pilih tepat satu pilihan. Jawab dengan A atau B sahaja.'
}
NOTE = {
    'en':{True:'Clinical note: Evidence of {p} is present.',False:'Clinical note: Evidence of {p} is absent.'},
    'id':{True:'Catatan klinis: Terdapat {p}.',False:'Catatan klinis: Tidak terdapat {p}.'},
    'ms':{True:'Nota klinikal: Terdapat {p}.',False:'Nota klinikal: Tiada {p}.'}
}

def norm_yes(x):
    x = str(x).strip().lower()
    if x in {'yes','ya','true','1'}: return True
    if x in {'no','tidak','false','0'}: return False
    raise ValueError(f'Non-binary reference answer: {x}')

def option_order(case_id, language):
    bit = int(hashlib.sha256(f'{SEED}|{case_id}|{language}'.encode()).hexdigest(),16) % 2
    return ['yes','no'] if bit == 0 else ['no','yes']

def build_prompt(row, language, condition):
    reference_yes = norm_yes(row.reference_answer)
    order = option_order(row.case_id, language)
    yn = {'yes':YESNO[language][0], 'no':YESNO[language][1]}
    parts = [INTRO[language]]
    note_yes = None
    if condition != 'image_only':
        note_yes = reference_yes if condition == 'concordant' else not reference_yes
        parts.append(NOTE[language][note_yes].format(p=row[PROP_COL[language]].strip()))
    parts += [row[QUESTION_COL[language]].strip(), f'A. {yn[order[0]]}\nB. {yn[order[1]]}', CHOOSE[language]]
    correct_letter = 'A' if order[0] == ('yes' if reference_yes else 'no') else 'B'
    note_letter = None if note_yes is None else ('A' if order[0] == ('yes' if note_yes else 'no') else 'B')
    return '\n\n'.join(parts), correct_letter, note_letter, order

def resolve_image(row):
    case_id = row.get("case_id", None)

    if case_id is None or pd.isna(case_id) or not str(case_id).strip():
        case_id = row.name

    case_id = str(case_id).strip()
    dataset = str(row.get("dataset", "")).strip()

    # Ollama-compatible transport copy for the problematic image
    if case_id == "C072":
        safe_path = ROOT / "images" / "vqa_rad" / "C072_ollama_safe.jpg"

        if safe_path.is_file():
            return safe_path

        raise FileNotFoundError(
            f"Normalized C072 image is missing: {safe_path}"
        )

    candidates = []

    recorded_path = row.get("image_path", "")
    if pd.notna(recorded_path) and str(recorded_path).strip():
        recorded_path = Path(str(recorded_path).strip())

        if recorded_path.is_absolute():
            candidates.append(recorded_path)
        else:
            candidates.append(ROOT / recorded_path)

    candidates.extend([
        ROOT / "images" / dataset / f"{case_id}.png",
        ROOT / "images" / dataset / f"{case_id}.jpg",
        ROOT / "images" / dataset / f"{case_id}.jpeg",
    ])

    for path in candidates:
        if path.is_file():
            return path

    raise FileNotFoundError(
        f"Image not found for case_id={case_id}, dataset={dataset}. "
        f"Checked: {[str(path) for path in candidates]}"
    )

for _, row in cases.iterrows(): resolve_image(row)
print('Resolved all',len(cases),'images.')
print('Locked prompt example:\n',build_prompt(cases.iloc[0],'en','conflicting')[0])


Resolved all 125 images.
Locked prompt example:
 Examine the medical image and answer the question.

Clinical note: Evidence of enhancing lesion is absent.

is the lesion enhancing?

A. Yes
B. No

Choose exactly one option. Reply with A or B only.


## 4. Checkpointed inference engine

An `ok` record is never repeated. Error records are retried on the next run. Parse failures remain recorded outcomes rather than being selectively regenerated.


In [28]:
def api_json(path, payload=None, timeout=3600):
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = urllib.request.Request('http://127.0.0.1:11434'+path,data=data,headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=timeout) as response:
        return json.loads(response.read())

def model_slug(model): return re.sub(r'[^a-z0-9]+','_',model.lower()).strip('_')
def checkpoint_path(model): return RESULTS/f'main_{model_slug(model)}.jsonl'

def append_jsonl(path,record):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(record,ensure_ascii=False)+'\n')
        f.flush(); os.fsync(f.fileno())

def load_records(model):
    path = checkpoint_path(model)
    if not path.exists(): return []
    out=[]
    for i,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
        try: out.append(json.loads(line))
        except json.JSONDecodeError: print(f'Ignoring incomplete line {i} in {path.name}')
    return out

def parse_letter(text):
    m = re.search(r'(?<![A-Z])[AB](?![A-Z])',str(text).strip().upper())
    return m.group(0) if m else None

def model_metadata(model):
    info = api_json('/api/show',{'model':model})
    tags = api_json('/api/tags').get('models',[])
    tag = next((x for x in tags if x.get('name')==model or x.get('model')==model),{})
    return {'model':model,'digest':tag.get('digest',''),'details':tag.get('details',{}),'modelfile_sha256':hashlib.sha256(info.get('modelfile','').encode()).hexdigest()}

def trial_id(model,case_id,language,condition): return f'{model}|{case_id}|{language}|{condition}'

def results_table(model):
    rows=load_records(model)
    if not rows: return pd.DataFrame()
    d=pd.DataFrame(rows)
    ok=d[d.status.eq('ok')].drop_duplicates('trial_id',keep='last')
    ok.to_csv(RESULTS/f'main_{model_slug(model)}_latest.csv',index=False)
    return ok

def run_model(model, limit=None, retries=2):
    meta=model_metadata(model)
    prior=load_records(model)
    prior_ok=[r for r in prior if r.get('status')=='ok']
    bad_hash=[r.get('trial_id') for r in prior_ok if r.get('casebook_sha256')!=actual_hash or r.get('schedule_sha256')!=schedule_sha]
    bad_digest=[r.get('trial_id') for r in prior_ok if r.get('model_digest')!=meta['digest']]
    if bad_hash: raise RuntimeError(f'Checkpoint protocol mismatch in {len(bad_hash)} completed trials.')
    if bad_digest: raise RuntimeError(f'Model digest changed; refusing to mix versions ({len(bad_digest)} prior trials).')
    done={r['trial_id'] for r in prior_ok}
    todo=schedule.copy()
    if limit is not None: todo=todo.iloc[:limit]
    case_lookup=cases.set_index('case_id')
    total=len(todo)
    for n,s in enumerate(todo.itertuples(index=False),1):
        row=case_lookup.loc[s.case_id]
        tid=trial_id(model,s.case_id,s.language,s.condition)
        if tid in done:
            print(f'[{n}/{total}] skip {tid}')
            continue
        row = row.copy()
        row["case_id"] = str(s.case_id)
        image_path=resolve_image(row)
        image_b64=base64.b64encode(image_path.read_bytes()).decode('ascii')
        prompt,correct_letter,note_letter,order=build_prompt(row,s.language,s.condition)
        payload={'model':model,'stream':False,'keep_alive':'10m','messages':[{'role':'user','content':prompt,'images':[image_b64]}],'options':{'temperature':0,'seed':SEED,'num_predict':4}}
        for attempt in range(1,retries+2):
            started=time.time()
            try:
                response=api_json('/api/chat',payload)
                raw=response.get('message',{}).get('content','')
                parsed=parse_letter(raw)
                record={
                    'trial_id':tid,'status':'ok','timestamp_utc':datetime.now(timezone.utc).isoformat(),
                    'model':model,'model_digest':meta['digest'],'model_details':meta['details'],
                    'casebook_sha256':actual_hash,'schedule_sha256':schedule_sha,'case_id':s.case_id,
                    'dataset':row.dataset,'language':s.language,'condition':s.condition,
                    'reference_answer':row.reference_answer,'option_a':order[0],'option_b':order[1],
                    'correct_letter':correct_letter,'note_letter':note_letter,'raw_response':raw,
                    'parsed_letter':parsed,'is_correct':None if parsed is None else parsed==correct_letter,
                    'follows_note':None if note_letter is None or parsed is None else parsed==note_letter,
                    'prompt_sha256':hashlib.sha256(prompt.encode()).hexdigest(),
                    'image_sha256':sha256_file(image_path),'elapsed_seconds':round(time.time()-started,3),
                    'attempt':attempt,'eval_count':response.get('eval_count'),'total_duration_ns':response.get('total_duration')
                }
                append_jsonl(checkpoint_path(model),record); done.add(tid)
                print(f'[{n}/{total}] {tid} -> {raw!r} ({record["elapsed_seconds"]}s)')
                break
            except Exception as e:
                error={'trial_id':tid,'status':'error','timestamp_utc':datetime.now(timezone.utc).isoformat(),'model':model,'case_id':s.case_id,'language':s.language,'condition':s.condition,'attempt':attempt,'error_type':type(e).__name__,'error':str(e),'elapsed_seconds':round(time.time()-started,3)}
                append_jsonl(checkpoint_path(model),error)
                print(f'[{n}/{total}] attempt {attempt} failed: {e}')
                if attempt<=retries: time.sleep(5*attempt)
    return results_table(model)

print('Checkpoint engine ready. Durable directory:',RESULTS)


Checkpoint engine ready. Durable directory: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results/main_evaluation


## 5. Qwen preflight (18 trials that count toward the main evaluation)

These are the first 18 randomized schedule entries. They remain checkpointed and will be skipped by the full run.


In [16]:
QWEN_MODEL='qwen2.5vl:3b'
subprocess.run([OLLAMA,'pull',QWEN_MODEL],check=True)
print(json.dumps(model_metadata(QWEN_MODEL),indent=2))
qwen_preflight=run_model(QWEN_MODEL,limit=18)
display(qwen_preflight.tail(18)[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Qwen completed unique trials:',len(qwen_preflight),'/ 1125')


{
  "model": "qwen2.5vl:3b",
  "digest": "fb90415cde1ef08aa669ae74b082d49b158729b6db1ab183c941417d507e71a1",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "qwen25vl",
    "families": [
      "qwen25vl"
    ],
    "parameter_size": "3.8B",
    "quantization_level": "Q4_K_M",
    "context_length": 128000,
    "embedding_length": 2048
  },
  "modelfile_sha256": "ad0ce6d9b08eafbc4d22401cdbb11d071d700bae66bc3420d9ec45e8e0d0d11e"
}
[1/18] qwen2.5vl:3b|C128|en|concordant -> 'A' (114.646s)
[2/18] qwen2.5vl:3b|C127|id|image_only -> 'A. Ya' (1.367s)
[3/18] qwen2.5vl:3b|C020|ms|concordant -> 'B.' (1.512s)
[4/18] qwen2.5vl:3b|C084|id|conflicting -> 'A.' (1.344s)
[5/18] qwen2.5vl:3b|C065|en|conflicting -> 'B.' (1.317s)
[6/18] qwen2.5vl:3b|C031|id|concordant -> 'A' (2.958s)
[7/18] qwen2.5vl:3b|C067|ms|concordant -> 'B. Tidak' (1.306s)
[8/18] qwen2.5vl:3b|C105|en|image_only -> 'A. No' (1.473s)
[9/18] qwen2.5vl:3b|C031|ms|image_only -> 'B. Ya' (0.211s)
[10/18] qwen2.5vl:3b

,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C128,en,concordant,A,A,True,True,114.646
1,C127,id,image_only,A. Ya,A,True,None,1.367
2,C020,ms,concordant,B.,B,True,True,1.512
3,C084,id,conflicting,A.,A,False,True,1.344
4,C065,en,conflicting,B.,B,False,True,1.317
5,C031,id,concordant,A,A,True,True,2.958
6,C067,ms,concordant,B. Tidak,B,True,True,1.306
7,C105,en,image_only,A. No,A,True,None,1.473
8,C031,ms,image_only,B. Ya,B,False,None,0.211
9,C064,en,concordant,B.,B,True,True,1.500


Qwen completed unique trials: 18 / 1125


## 6. Run or resume the complete Qwen evaluation

Safe to rerun after interruption.


In [ ]:
from PIL import Image, ImageOps
from pathlib import Path

C072_ORIGINAL = (
    ROOT / "images" / "vqa_rad" / "C072.jpg"
)
C072_SAFE = (
    ROOT / "images" / "vqa_rad" / "C072_ollama_safe.jpg"
)

with Image.open(C072_ORIGINAL) as image:
    image = ImageOps.exif_transpose(image).convert("RGB")

    print("Original dimensions:", image.size)

    # Reduce only if necessary while preserving aspect ratio
    image.thumbnail((1600, 1600), Image.Resampling.LANCZOS)

    image.save(
        C072_SAFE,
        format="JPEG",
        quality=95,
        subsampling=0,
    )

with Image.open(C072_SAFE) as image:
    print("Safe-copy dimensions:", image.size)
    print("Safe-copy mode:", image.mode)
    print("Safe-copy size:", C072_SAFE.stat().st_size, "bytes")
    display(image)

print("Created:", C072_SAFE)

In [29]:
qwen_results=run_model(QWEN_MODEL)
print('Qwen completed unique trials:',len(qwen_results),'/ 1125')
assert len(qwen_results)==1125,'Qwen is incomplete; rerun this cell to retry unfinished/error trials.'
print('Qwen parse failures:',int(qwen_results['parsed_letter'].isna().sum()))


[1/1125] skip qwen2.5vl:3b|C128|en|concordant
[2/1125] skip qwen2.5vl:3b|C127|id|image_only
[3/1125] skip qwen2.5vl:3b|C020|ms|concordant
[4/1125] skip qwen2.5vl:3b|C084|id|conflicting
[5/1125] skip qwen2.5vl:3b|C065|en|conflicting
[6/1125] skip qwen2.5vl:3b|C031|id|concordant
[7/1125] skip qwen2.5vl:3b|C067|ms|concordant
[8/1125] skip qwen2.5vl:3b|C105|en|image_only
[9/1125] skip qwen2.5vl:3b|C031|ms|image_only
[10/1125] skip qwen2.5vl:3b|C064|en|concordant
[11/1125] skip qwen2.5vl:3b|C051|ms|concordant
[12/1125] skip qwen2.5vl:3b|C089|ms|image_only
[13/1125] skip qwen2.5vl:3b|C126|en|conflicting
[14/1125] skip qwen2.5vl:3b|C029|ms|conflicting
[15/1125] skip qwen2.5vl:3b|C074|en|conflicting
[16/1125] skip qwen2.5vl:3b|C019|id|conflicting
[17/1125] skip qwen2.5vl:3b|C015|en|concordant
[18/1125] skip qwen2.5vl:3b|C042|en|image_only
[19/1125] skip qwen2.5vl:3b|C069|id|image_only
[20/1125] skip qwen2.5vl:3b|C114|ms|image_only
[21/1125] skip qwen2.5vl:3b|C068|id|image_only
[22/1125] skip q

### TroubleShooting

In [ ]:
# Find the 1125-row trial schedule automatically
schedule_candidates = []

for name, obj in globals().items():
    if (
        isinstance(obj, pd.DataFrame)
        and len(obj) == 1125
        and {"case_id", "language", "condition"}.issubset(obj.columns)
    ):
        schedule_candidates.append((name, obj))

print("Schedule candidates:", [name for name, _ in schedule_candidates])

if not schedule_candidates:
    raise RuntimeError(
        "Could not locate the 1125-row schedule. "
        "Please show the output of: print(list(globals().keys()))"
    )

schedule_name, expected_trials = schedule_candidates[0]
expected_trials = expected_trials[
    ["case_id", "language", "condition"]
].astype(str).drop_duplicates()

completed_trials = qwen_results[
    ["case_id", "language", "condition"]
].astype(str).drop_duplicates()

missing_trials = (
    expected_trials.merge(
        completed_trials,
        on=["case_id", "language", "condition"],
        how="left",
        indicator=True,
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
    .sort_values(["case_id", "language", "condition"])
)

print("Schedule used:", schedule_name)
print("Missing trials:", len(missing_trials))
display(missing_trials)

print("\nMissing trials per case:")
display(
    missing_trials.groupby("case_id")
    .size()
    .rename("missing_count")
    .reset_index()
)

In [23]:
print("Possible Qwen/checkpoint variables:")

for name, value in list(globals().items()):
    lowered = name.lower()

    if "qwen" in lowered or "checkpoint" in lowered or lowered.endswith("_cp"):
        if isinstance(value, (str, Path)):
            print(name, "=", value)

Possible Qwen/checkpoint variables:
QWEN_MODEL = qwen2.5vl:3b


In [ ]:
import json
from pathlib import Path

print("Searching JSONL checkpoints for C072 errors...\n")

found = 0

for path in ROOT.rglob("*.jsonl"):
    try:
        with path.open("r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                if "C072" not in line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                if record.get("status") == "error":
                    found += 1
                    print("FILE:", path)
                    print("LINE:", line_number)
                    print(json.dumps(record, indent=2, ensure_ascii=False))
                    print()
    except Exception:
        pass

print("C072 error records found:", found)

In [ ]:
from PIL import Image

c072_rows = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    if "case_id" in obj.columns:
        matched = obj[obj["case_id"].astype(str).eq("C072")]
        if len(matched):
            c072_rows.append((name, matched.iloc[0]))

    elif obj.index.name == "case_id" and "C072" in obj.index.astype(str):
        c072_rows.append((name, obj.loc["C072"]))

print("Dataframes containing C072:", [name for name, _ in c072_rows])

if not c072_rows:
    raise RuntimeError("Could not locate C072 in an active dataframe.")

source_name, c072_row = c072_rows[0]
c072_row = c072_row.copy()
c072_row["case_id"] = "C072"

c072_path = resolve_image(c072_row)

print("Source dataframe:", source_name)
print("Image path:", c072_path)
print("File size:", c072_path.stat().st_size, "bytes")

with Image.open(c072_path) as image:
    print("Format:", image.format)
    print("Mode:", image.mode)
    print("Dimensions:", image.size)
    image.verify()

print("Pillow verification: PASSED")
display(Image.open(c072_path))

## 7. Unload Qwen and run Gemma preflight


In [30]:
try: api_json('/api/generate',{'model':QWEN_MODEL,'keep_alive':0})
except Exception as e: print('Unload note:',e)
GEMMA_MODEL='gemma3:4b'
subprocess.run([OLLAMA,'pull',GEMMA_MODEL],check=True)
print(json.dumps(model_metadata(GEMMA_MODEL),indent=2))
gemma_preflight=run_model(GEMMA_MODEL,limit=18)
display(gemma_preflight.tail(18)[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Gemma completed unique trials:',len(gemma_preflight),'/ 1125')


{
  "model": "gemma3:4b",
  "digest": "a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "gemma3",
    "families": [
      "gemma3"
    ],
    "parameter_size": "4.3B",
    "quantization_level": "Q4_K_M",
    "context_length": 131072,
    "embedding_length": 2560
  },
  "modelfile_sha256": "5e30e95594473dde0d7779928fee0de00266fd86934b65645af7e93fe83b9fda"
}
[1/18] gemma3:4b|C128|en|concordant -> 'A\n' (55.509s)
[2/18] gemma3:4b|C127|id|image_only -> 'A\n' (1.428s)
[3/18] gemma3:4b|C020|ms|concordant -> 'B. Tidak\n' (1.399s)
[4/18] gemma3:4b|C084|id|conflicting -> 'A\n' (1.38s)
[5/18] gemma3:4b|C065|en|conflicting -> 'B' (1.364s)
[6/18] gemma3:4b|C031|id|concordant -> 'A\n' (1.415s)
[7/18] gemma3:4b|C067|ms|concordant -> 'B\n' (1.387s)
[8/18] gemma3:4b|C105|en|image_only -> 'B\n' (1.357s)
[9/18] gemma3:4b|C031|ms|image_only -> 'B. Ya' (1.454s)
[10/18] gemma3:4b|C064|en|concordant -> 'B' (1.402s)


,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C128,en,concordant,A\n,A,True,True,55.509
1,C127,id,image_only,A\n,A,True,None,1.428
2,C020,ms,concordant,B. Tidak\n,B,True,True,1.399
3,C084,id,conflicting,A\n,A,False,True,1.380
4,C065,en,conflicting,B,B,False,True,1.364
5,C031,id,concordant,A\n,A,True,True,1.415
6,C067,ms,concordant,B\n,B,True,True,1.387
7,C105,en,image_only,B\n,B,False,None,1.357
8,C031,ms,image_only,B. Ya,B,False,None,1.454
9,C064,en,concordant,B,B,True,True,1.402


Gemma completed unique trials: 18 / 1125


## 8. Run or resume the complete Gemma evaluation


In [31]:
gemma_results=run_model(GEMMA_MODEL)
print('Gemma completed unique trials:',len(gemma_results),'/ 1125')
assert len(gemma_results)==1125,'Gemma is incomplete; rerun this cell to retry unfinished/error trials.'
print('Gemma parse failures:',int(gemma_results['parsed_letter'].isna().sum()))


[1/1125] skip gemma3:4b|C128|en|concordant
[2/1125] skip gemma3:4b|C127|id|image_only
[3/1125] skip gemma3:4b|C020|ms|concordant
[4/1125] skip gemma3:4b|C084|id|conflicting
[5/1125] skip gemma3:4b|C065|en|conflicting
[6/1125] skip gemma3:4b|C031|id|concordant
[7/1125] skip gemma3:4b|C067|ms|concordant
[8/1125] skip gemma3:4b|C105|en|image_only
[9/1125] skip gemma3:4b|C031|ms|image_only
[10/1125] skip gemma3:4b|C064|en|concordant
[11/1125] skip gemma3:4b|C051|ms|concordant
[12/1125] skip gemma3:4b|C089|ms|image_only
[13/1125] skip gemma3:4b|C126|en|conflicting
[14/1125] skip gemma3:4b|C029|ms|conflicting
[15/1125] skip gemma3:4b|C074|en|conflicting
[16/1125] skip gemma3:4b|C019|id|conflicting
[17/1125] skip gemma3:4b|C015|en|concordant
[18/1125] skip gemma3:4b|C042|en|image_only
[19/1125] gemma3:4b|C069|id|image_only -> 'A' (1.48s)
[20/1125] gemma3:4b|C114|ms|image_only -> 'A. Ya' (1.425s)
[21/1125] gemma3:4b|C068|id|image_only -> 'A' (1.362s)
[22/1125] gemma3:4b|C089|ms|conflicting -> 

## 9. Completeness gate and descriptive outputs

This is a data-quality summary only. Formal paired statistical analysis belongs in the next notebook.


In [32]:
frames=[]
for model in [QWEN_MODEL,GEMMA_MODEL]:
    d=results_table(model)
    print(model,len(d),'/ 1125 successful unique responses')
    assert len(d)==1125,f'{model} incomplete.'
    assert d['trial_id'].is_unique
    frames.append(d)

all_results=pd.concat(frames,ignore_index=True)
assert len(all_results)==2250 and all_results['trial_id'].is_unique
all_results.to_csv(RESULTS/'main_all_trials.csv',index=False)

summary=(all_results.groupby(['model','language','condition'],dropna=False)
         .agg(n=('trial_id','size'),accuracy=('is_correct','mean'),
              note_following=('follows_note','mean'),median_seconds=('elapsed_seconds','median'),
              parse_failures=('parsed_letter',lambda x:int(x.isna().sum())))
         .reset_index())
summary.to_csv(RESULTS/'main_descriptive_summary.csv',index=False)

completeness=(all_results.groupby(['model','language','condition']).size().rename('n').reset_index())
assert completeness['n'].eq(125).all(),completeness
completeness.to_csv(RESULTS/'main_completeness.csv',index=False)

manifest={
    'completed_at_utc':datetime.now(timezone.utc).isoformat(),
    'casebook_sha256':actual_hash,'schedule_sha256':schedule_sha,
    'models':{m:model_metadata(m) for m in [QWEN_MODEL,GEMMA_MODEL]},
    'n_cases':125,'n_trials_per_model':1125,'n_trials_total':2250,
    'languages':LANGUAGES,'conditions':CONDITIONS,'seed':SEED
}
(RESULTS/'main_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
display(summary)
print('Main evaluation complete. Durable outputs:',RESULTS)


qwen2.5vl:3b 1125 / 1125 successful unique responses
gemma3:4b 1125 / 1125 successful unique responses


,model,language,condition,n,accuracy,note_following,median_seconds,parse_failures
0,gemma3:4b,en,concordant,125,0.984,0.984,1.486,0
1,gemma3:4b,en,conflicting,125,0.04,0.96,1.486,0
2,gemma3:4b,en,image_only,125,0.6,NaN,1.485,0
3,gemma3:4b,id,concordant,125,0.984,0.984,1.495,0
4,gemma3:4b,id,conflicting,125,0.048,0.952,1.489,0
5,gemma3:4b,id,image_only,125,0.616,NaN,1.480,0
6,gemma3:4b,ms,concordant,125,0.968,0.968,1.502,0
7,gemma3:4b,ms,conflicting,125,0.08,0.92,1.503,0
8,gemma3:4b,ms,image_only,125,0.6,NaN,1.486,0
9,qwen2.5vl:3b,en,concordant,125,0.992,0.992,0.178,0


Main evaluation complete. Durable outputs: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results/main_evaluation


## Stop here

Preserve the JSONL checkpoints. The next notebook will perform paired case-level analysis, confidence intervals, cross-language invariance tests, dataset replication comparisons, and compact tables/figures for the Research Letter.
